In [1]:
import folium, math
import pandas as pd
import geopandas as gpd
import branca.colormap as cm
import folium.plugins as plugins


In [3]:
homes_tao = pd.read_parquet("/glade/derecho/scratch/ggantos/CAT/home_locations_tao_nofilters/home_loc_by_row_group.parquet")
homes_org = pd.read_parquet("/glade/derecho/scratch/cbecker/CAT/home_locations/home_loc_by_row_group.parquet")

In [4]:
print(homes_tao.shape)
print(homes_org.shape)


(34603918, 8)
(57379828, 4)


In [5]:
survey_df = pd.read_csv("../Datasets/final_data/2024_AR_longitudinal_final.csv")
affected_zips = [str(int(x)) for x in survey_df["inputzip"].unique() if not math.isnan(x)]

In [6]:
gdf_zips = gpd.read_file("/glade/work/cbecker/california_zip.geojson")
gdf_zips_count = pd.merge(gdf_zips, homes_tao.groupby("zipcode").size().rename("count_tao"), left_on="ZIP_CODE", right_on="zipcode")
# gdf_zips_count = pd.merge(gdf_zips_count, homes_skm.groupby("zipcode").size().rename("count_skm"), left_on="ZIP_CODE", right_on="zipcode")
gdf_zips_count = pd.merge(gdf_zips_count, homes_org.groupby("zipcode").size().rename("count_org"), left_on="ZIP_CODE", right_on="zipcode")
gdf_zips_count = gdf_zips_count[gdf_zips_count.ZIP_CODE.isin(affected_zips)]
gdf_zips_count.shape

(550, 10)

In [ ]:
BuPu_07 = ["#edf8fb","#bfd3e6","#9ebcda","#8c96c6","#8c6bb1","#88419d","#6e016b"]
# count_cols = ["count_tao", "count_skm", "count_org"]
count_cols = ["count_tao", "count_org"]
tooltips = ["ZIP_CODE", "POPULATION", "POP_SQMI"]
aliases = ["Zip Code:", "Population:", "Square Miles:", "# Homes:"]
vmax = gdf_zips_count[count_cols].max().max()
linear = cm.LinearColormap(BuPu_07,
                           caption= "Number of Homes",
                           vmin=1,
                           vmax=vmax,
                          ).to_step(index=[1, 100000, 200000, 500000, 1000000, 2000000, 2500000],
                                   )

m = plugins.DualMap(location=[34.0549, -118.2426],
                    zoom_start=8,
                    control_scale=True,
                   )

for count_col, mcol in zip(count_cols, [m.m1, m.m2]):
    
    tooltip = folium.GeoJsonTooltip(
        fields=tooltips+[count_col],
        aliases=aliases,
        localize=True,
        sticky=True,
        labels=True,
        style="""
            background-color: #F0EFEF;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
        max_width=800,
    )
    
    popup = folium.GeoJsonPopup(
        fields=tooltips+[count_col],
        aliases=aliases,
        localize=True,
        sticky=True,
        labels=True,
        style="""
            background-color: #F0EFEF;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
        max_width=800,
    )
    
    folium.GeoJson(
        gdf_zips_count,
        # zoom_on_click=True,
        style_function=lambda feature: {
            "fillColor": linear(feature["properties"][count_col]),
            "fillOpacity": 0.6,
            "color": "black",
            "weight": 0.5,
        },
        highlight_function=lambda feature: {
            "fillColor": "#78e05b",
            "fillOpacity": 0.6,
            "color": "black",
            "weight": 0.5,
        },
        popup=popup,
        popup_keep_highlighted=True,
        tooltip=tooltip,
        name=count_col,
    ).add_to(mcol)


linear.add_to(m.m2)
plugins.Geocoder().add_to(m.m2)
plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m.m2)
folium.LayerControl().add_to(m)

m.save("html/home_locations_comp.html")
m

![title](img/home_loc_dual.png)